# Incrementality & Causal ROI
Observational incrementality readout for paid and email channels.

**Notebook goals**
- Load saved incrementality artifacts (dataset, lift tables, ROI summary).
- Compare naive vs causal ROI for paid/email channels.
- Review propensity overlap & balance diagnostics for stakeholder decks.
- Capture next actions for marketing leaders.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'

incrementality_df = pd.read_csv(DATA_DIR / 'incrementality_dataset.csv')
roi_summary = pd.read_csv(DATA_DIR / 'incremental_roi_summary.csv')
psm_results = pd.read_csv(DATA_DIR / 'psm_att_results.csv')
ipw_results = pd.read_csv(DATA_DIR / 'ipw_ate_results.csv')
naive_incrementality = pd.read_csv(DATA_DIR / 'naive_incrementality_metrics.csv')

incrementality_df.shape, roi_summary.shape

## Naive vs causal ROI
Channel-level ROI deltas using heuristic attribution vs causal estimates.

In [ ]:
roi_long = roi_summary.rename(columns={'treatment': 'channel'}).copy()
naive_subset = (
    naive_incrementality
    .rename(
        columns={
            'treatment': 'channel',
            'lift_rev': 'naive_lift_rev',
            'incremental_ROI': 'naive_incremental_ROI',
            'incremental_ROAS': 'naive_incremental_ROAS'
        }
    )
    [['channel', 'naive_lift_rev', 'naive_incremental_ROI', 'naive_incremental_ROAS']]
)

roi_comparison = (
    roi_long
    .merge(naive_subset, on='channel', how='left')
    .sort_values(['channel', 'method'])
)
roi_comparison

## Lift estimates by method
Side-by-side comparison of PSM (ATT) and weighting (ATE/AIPW) outcomes.

In [ ]:
lift_long = pd.concat([psm_results, ipw_results], ignore_index=True)
lift_pivot = (
    lift_long
    .pivot_table(index=['treatment', 'method'], columns='outcome', values='estimate')
    .reset_index()
)
lift_pivot

## Diagnostics
Propensity overlap (top) and standardized mean differences (bottom).

In [ ]:
display(Image(filename=str(FIG_DIR / 'propensity_overlap.png')))
display(Image(filename=str(FIG_DIR / 'psm_balance_plot.png')))

## Decision highlights

In [ ]:
paid_att = (
    psm_results
    .query("treatment == 'treated_paid' and outcome == 'y_conv'")
    ['estimate']
    .iat[0]
)
email_att = (
    psm_results
    .query("treatment == 'treated_email' and outcome == 'y_conv'")
    ['estimate']
    .iat[0]
)
paid_ipw_rev = (
    ipw_results
    .query("treatment == 'treated_paid' and method == 'IPW_ATE' and outcome == 'y_rev'")
    ['estimate']
    .iat[0]
)
email_aipw_rev = (
    ipw_results
    .query("treatment == 'treated_email' and method == 'AIPW' and outcome == 'y_rev'")
    ['estimate']
    .iat[0]
)
points = [
    f"- Paid ATT lift: {paid_att:.2%} conversion-rate bump among exposed journeys.",
    f"- Email ATT lift: {email_att:.2%} conversion-rate bump; nurture stays incremental.",
    f"- Paid IPW revenue lift: ${paid_ipw_rev:,.0f} per user; monitor negative or flat estimates.",
    f"- Email AIPW revenue lift: ${email_aipw_rev:,.0f} per user, reinforcing lifecycle triggers.",
    "- Prioritize tests where overlap/balance are strongest and budget can be shifted quickly.",
]
display(Markdown('\n'.join(points)))

## What to do next
- Spin up geo or audience-level holdout for paid to validate the negative incremental ROI.
- Double down on email retargeting creatives and experiment with incremental budget.
- Refresh propensity features quarterly to keep overlap healthy.
- Fold this notebook output into the exec dashboard after each pipeline run.